# CI integration test: basic example

Mirrors `examples/mneflow_example_tf2.ipynb`'s pipeline (import -> tfrecords
-> LFCNN -> train -> evaluate -> save), but always runs on a small synthetic
dataset with a handful of training epochs, so it's fast and needs no
download. Not a tutorial -- see the notebook in `examples/` for that.

Writes its tfrecords + trained model to `MNEFLOW_DATA_PATH` (a temp dir in
CI) so `save_restore_ci.ipynb` and `own_graph_example_ci.ipynb` can reload
them, exactly as the real examples chain off of the basic example.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import mne
mne.set_log_level(verbose='CRITICAL')

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import mneflow
print(mneflow.__version__)

In [ ]:
n_epochs = int(os.environ.get('MNEFLOW_N_EPOCHS', '3'))
path = os.environ.get('MNEFLOW_DATA_PATH', '/tmp/mneflow_ci/')
data_id = 'mne_sample_multimodal'

In [ ]:
# Synthetic stand-in for the real multimodal dataset used in
# examples/mneflow_example_tf2.ipynb: ~200 grad channels, 8 pseudo-conditions,
# matching shape/sfreq, no download.
rng = np.random.RandomState(0)
ch_names = [f'MEG{i:04d}' for i in range(204)]
info = mne.create_info(ch_names, 600., ch_types='grad')
condition_counts = {
    'Visual Upper right': 117, 'Visual Lower right': 129,
    'Visual Lower left': 115, 'Visual Upper left': 133,
    'Somato right': 107, 'Somato left': 118,
    'Auditory right': 104, 'Auditory left': 117,
}
event_id = {name: i + 1 for i, name in enumerate(condition_counts)}
n_total = sum(condition_counts.values())
n_samples = 421  # matches tmin=-0.0998976, tmax=0.499488 @ 600Hz
data = (rng.randn(n_total, 204, n_samples) * 1e-12).astype(np.float32)
labels = np.concatenate([[event_id[name]] * cnt for name, cnt in condition_counts.items()])
rng.shuffle(labels)
events = np.zeros((n_total, 3), dtype=int)
events[:, 0] = np.arange(n_total) * 1000
events[:, 2] = labels
epochs = mne.EpochsArray(data, info, events=events, tmin=-0.0998976, event_id=event_id, verbose=False)
epochs = epochs.pick_types(meg='grad')
print(epochs)

In [ ]:
import_opt = dict(path=path,
                  data_id=data_id,
                  input_type='trials',
                  target_type='int',
                  n_folds=5,
                  test_set='holdout',
                  fs=600,
                  overwrite=True,
                  scale=True,
                  crop_baseline=True,
                  scale_interval=(0, 60),
                  )

meta = mneflow.produce_tfrecords(epochs, **import_opt)

In [ ]:
dataset = mneflow.Dataset(meta, train_batch=100, class_subset=[0, 1, 2, 3, 4, 5, 6])

lfcnn_params = dict(n_latent=32,
                  filter_length=17,
                  nonlin=tf.nn.relu,
                  padding='SAME',
                  pooling=5,
                  stride=5,
                  pool_type='max',
                  dropout=.5,
                  l1_scope=["weights"],
                  l1=3e-3)

meta.update(model_specs=lfcnn_params)

model = mneflow.LFCNN(meta)
model.build()

In [ ]:
model.train(n_epochs=n_epochs, eval_step=5, early_stopping=3, mode='single_fold')

In [ ]:
test_loss, test_acc = model.evaluate(meta.data['test_paths'])
print("Test set: Loss = {:.4f} Accuracy = {:.4f}".format(test_loss, test_acc))

In [ ]:
meta.save()
model.save()